# Défauts de paiement de crédit : évaluation et explicabilité

Une banque veut prédire lesquels de ses clients de carte de crédit feront défaut le mois suivant, pour refuser ou réduire leur crédit, et pouvoir expliquer chaque décision, au client comme au régulateur : l'évaluation de la solvabilité est un système à haut risque de l'AI Act. Les données : 30 000 clients d'une banque taïwanaise en 2005 ([Yeh & Lien, 2009](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients), CC BY 4.0), dont 22 % ont fait défaut.

1. Un premier modèle et ses aires sous les courbes
2. Le seuil de décision face à une matrice de coût
3. La calibration des probabilités
4. Explication globale avec SHAP
5. Un client mal prédit face à un client bien prédit
6. LIME face à SHAP sur le même client
7. Importance par permutation et dépendance partielle
8. Tranches des données : où le modèle coûte le plus
9. Équité : les erreurs par groupe

Le notebook tourne sans GPU, en quelques minutes.

## Données

Chaque ligne est un client : `limit` (le crédit accordé, en dollars taïwanais), `sex` (1 = homme, 2 = femme), `education` (1 = 2e ou 3e cycle universitaire, 2 = université, 3 = lycée, 4 = autre ; 0, 5 et 6 ne sont pas documentés), `marriage`, `age`, puis pour chaque mois d'avril à septembre 2005 : `delay_<mois>` (le retard de paiement, en mois ; −1 : payé à temps ; −2 et 0, non documentés, ne sont pas des retards), `bill_<mois>` (le montant du relevé) et `paid_<mois>` (le montant payé). `default` vaut 1 si le client a fait défaut le mois suivant. Le [README du jeu de données](https://github.com/shuuchuu/datasets/tree/main/credit-default) détaille les colonnes.

In [ ]:
# lime et fairlearn manquent à Colab ; son shap 0.52 refuse un jeu de fond avec xgboost 3
%pip install -q lime fairlearn "shap>=0.53"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh credit-default

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

clients = pd.read_csv("credit-default/credit-default.csv", index_col="id")
# En flottants : la dépendance partielle de scikit-learn refuse les colonnes entières.
X = clients.drop(columns="default").astype(float)
y = clients["default"]
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(
  f"{len(X_train)} clients d'entraînement, {len(X_test)} de test, "
  f"{y.mean():.1%} de défauts"
)
clients.head()

## 1. Un premier modèle

Entraînez `model`, un `XGBClassifier`, sur `X_train` et `y_train`, et calculez `proba`, la probabilité de défaut de chaque client de `X_test`. Comparez l'exactitude du modèle au seuil de 0,5 à celle d'un modèle qui ne prédit jamais de défaut. Calculez ensuite l'AUC ROC et l'AUC PR : à quelle valeur, celle du hasard, comparer chacune ? Tracez enfin les deux courbes côte à côte.

**Documentation utile**

- [`accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)` : 300 arbres peu profonds, appris lentement ; `random_state` rend l'entraînement reproductible
- `model.predict_proba(X_test)[:, 1]` : la colonne de la classe 1, le défaut
- `accuracy_score(y_test, proba > 0.5)` ; un modèle qui ne prédit jamais de défaut a pour exactitude la part des clients sains, `1 - y_test.mean()`
- `roc_auc_score(y_test, proba)` et `average_precision_score(y_test, proba)` : le hasard vaut 0,5 pour la première, la prévalence `y_test.mean()` pour la seconde
- `fig, axes = plt.subplots(1, 2, figsize=(10, 4))`, puis `RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])`, et de même `PrecisionRecallDisplay` sur `axes[1]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import (
  PrecisionRecallDisplay,
  RocCurveDisplay,
  accuracy_score,
  average_precision_score,
  roc_auc_score,
)
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)
model.fit(..., ...)
proba = model.predict_proba(X_test)[:, ...]

print(f"Exactitude du modèle : {accuracy_score(y_test, ...):.3f}")
print(f"Exactitude de « jamais de défaut » : {...:.3f}")
print(f"AUC ROC : {roc_auc_score(y_test, proba):.3f} (hasard : 0,5)")
print(f"AUC PR : {...:.3f} (hasard : {y_test.mean():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(..., ..., ax=...)
plt.show()
```

### Solution

In [ ]:
from sklearn.metrics import (
  PrecisionRecallDisplay,
  RocCurveDisplay,
  accuracy_score,
  average_precision_score,
  roc_auc_score,
)
from xgboost import XGBClassifier

model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print(f"Exactitude du modèle : {accuracy_score(y_test, proba > 0.5):.3f}")
print(f"Exactitude de « jamais de défaut » : {1 - y_test.mean():.3f}")
print(f"AUC ROC : {roc_auc_score(y_test, proba):.3f} (hasard : 0,5)")
print(
  f"AUC PR : {average_precision_score(y_test, proba):.3f} "
  f"(hasard : {y_test.mean():.3f})"
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.show()

## 2. Le seuil de décision face à une matrice de coût

Refuser un crédit à un client qui l'aurait remboursé (un faux positif) fait perdre un client ; accorder un crédit à un client qui fait défaut (un faux négatif) coûte cinq fois plus. Écrivez `cost(y_true, y_pred)`, le coût total des erreurs, un faux positif coûtant `C_FP` et un faux négatif `C_FN`. Comparez le coût sur le jeu de test :

- au seuil de 0,5 ;
- au seuil `threshold` que donne la règle des slides pour des probabilités calibrées, $C_\text{FP} / (C_\text{FP} + C_\text{FN})$ ;
- au seuil que `TunedThresholdClassifierCV` choisit en validation croisée sur le jeu d'entraînement.

**Documentation utile**

- [`confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`make_scorer`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html)

In [ ]:
C_FP, C_FN = 1, 5  # le coût d'un faux positif, celui d'un faux négatif


def cost(y_true, y_pred) -> int: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()` renvoie `tn, fp, fn, tp` ; `labels=[0, 1]` garde une matrice 2 × 2 même sur un groupe de clients sans défaut
- Un exemple à vérifier à la main : `cost([0, 1, 1], [1, 0, 1])` vaut 6 (un faux positif, un faux négatif)
- `proba > threshold` : les refus à un seuil donné
- `make_scorer(cost, greater_is_better=False)` : un score à minimiser
- `TunedThresholdClassifierCV(estimator, scoring=...)`, avec `estimator` un modèle neuf, `XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0)`, puis `.fit(X_train, y_train)` (cinq entraînements, une dizaine de secondes) ; `tuned.best_threshold_` et `tuned.predict(X_test)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV

C_FP, C_FN = 1, 5


def cost(y_true, y_pred) -> int:
  _, fp, fn, _ = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
  return ...


print(f"Coût au seuil de 0,5 : {cost(y_test, proba > 0.5)}")
threshold = ...
print(f"Coût au seuil de {threshold:.3f} : {cost(y_test, proba > threshold)}")

tuned = TunedThresholdClassifierCV(
  XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0),
  scoring=make_scorer(..., greater_is_better=...),
)
tuned.fit(X_train, y_train)
print(
  f"Coût au seuil choisi en validation croisée ({tuned.best_threshold_:.3f}) : "
  f"{cost(y_test, tuned.predict(X_test))}"
)
```

### Solution

In [ ]:
from sklearn.metrics import confusion_matrix, make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV

C_FP, C_FN = 1, 5


def cost(y_true, y_pred) -> int:
  _, fp, fn, _ = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
  return int(C_FP * fp + C_FN * fn)


print(f"Coût au seuil de 0,5 : {cost(y_test, proba > 0.5)}")
threshold = C_FP / (C_FP + C_FN)
print(f"Coût au seuil de {threshold:.3f} : {cost(y_test, proba > threshold)}")

tuned = TunedThresholdClassifierCV(
  XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=0),
  scoring=make_scorer(cost, greater_is_better=False),
)
tuned.fit(X_train, y_train)
print(
  f"Coût au seuil choisi en validation croisée ({tuned.best_threshold_:.3f}) : "
  f"{cost(y_test, tuned.predict(X_test))}"
)

## 3. La calibration

La règle du seuil suppose des probabilités calibrées : parmi les clients à qui le modèle donne 0,2, 20 % doivent faire défaut. Tracez le diagramme de fiabilité de `proba` et calculez son score de Brier. Entraînez ensuite `balanced`, le même modèle, aux classes rééquilibrées par `scale_pos_weight` (le nombre de clients sains divisé par le nombre de défauts de `y_train`), et comparez les deux modèles : leur courbe, leur score de Brier, leur probabilité moyenne face au taux de défaut réel, et leur coût au seuil `threshold` puis à 0,5. Quel seuil faudrait-il au modèle rééquilibré ?

**Documentation utile**

- [`CalibrationDisplay.from_predictions`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.CalibrationDisplay.html#sklearn.calibration.CalibrationDisplay.from_predictions)
- [`brier_score_loss`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html)
- [`scale_pos_weight`](https://xgboost.readthedocs.io/en/stable/parameter.html#parameters-for-tree-booster)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `(y_train == 0).sum() / (y_train == 1).sum()` : environ 3,5
- `XGBClassifier(..., scale_pos_weight=ratio)`, avec les autres paramètres de `model`
- `CalibrationDisplay.from_predictions(y_test, proba, n_bins=10, name="XGBoost", ax=ax)` : la diagonale est la calibration parfaite ; un second appel sur le même `ax` y ajoute l'autre modèle
- `brier_score_loss(y_test, proba)` : l'erreur quadratique moyenne des probabilités, plus petite est meilleure

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss

ratio = ...
balanced = XGBClassifier(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=4,
  random_state=0,
  scale_pos_weight=...,
)
balanced.fit(X_train, y_train)
balanced_proba = balanced.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("XGBoost", proba), ("rééquilibré", balanced_proba)]:
  CalibrationDisplay.from_predictions(y_test, p, n_bins=10, name=name, ax=ax)
  print(
    f"{name} : Brier {...:.3f}, probabilité moyenne {p.mean():.3f}, "
    f"coût {cost(y_test, p > threshold)} à {threshold:.3f}, "
    f"{cost(y_test, p > 0.5)} à 0,5"
  )
print(f"Taux de défaut réel : {y_test.mean():.3f}")
plt.show()
```

### Solution

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss

ratio = (y_train == 0).sum() / (y_train == 1).sum()
balanced = XGBClassifier(
  n_estimators=300,
  learning_rate=0.05,
  max_depth=4,
  random_state=0,
  scale_pos_weight=ratio,
)
balanced.fit(X_train, y_train)
balanced_proba = balanced.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("XGBoost", proba), ("rééquilibré", balanced_proba)]:
  CalibrationDisplay.from_predictions(y_test, p, n_bins=10, name=name, ax=ax)
  print(
    f"{name} : Brier {brier_score_loss(y_test, p):.3f}, probabilité moyenne {p.mean():.3f}, "
    f"coût {cost(y_test, p > threshold)} à {threshold:.3f}, "
    f"{cost(y_test, p > 0.5)} à 0,5"
  )
print(f"Taux de défaut réel : {y_test.mean():.3f}")
plt.show()

## 4. Explication globale avec SHAP

Créez `explainer`, un `TreeExplainer` de `model` dont le jeu de fond est un échantillon de 100 clients d'entraînement, et calculez `explanation`, les valeurs SHAP de `sample`, les 1 000 premiers clients de test (TreeSHAP avec un jeu de fond y passe une dizaine de secondes). Vérifiez l'additivité : la valeur de base plus la somme des valeurs SHAP d'un client donne la sortie du modèle, en log-odds. Affichez le *beeswarm* et le graphique en barres : quelles variables comptent le plus, et dans quel sens agit `delay_sep` ?

**Documentation utile**

- [`shap.TreeExplainer`](https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html)
- [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)

In [ ]:
sample = X_test.iloc[:1000]

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `shap.TreeExplainer(model, X_train.sample(100, random_state=0))` : le jeu de fond est le second argument
- `explainer(sample)` renvoie un `shap.Explanation` : `.values` (une ligne par client, une colonne par variable) et `.base_values` (la valeur de base, la sortie moyenne du modèle sur le jeu de fond, la même pour tous)
- `model.predict(sample, output_margin=True)` : la sortie du modèle en log-odds ; l'écart maximal, `np.abs(...).max()`, doit être de l'ordre de 1e-6
- `shap.plots.beeswarm(explanation)` et `shap.plots.bar(explanation)`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import shap

sample = X_test.iloc[:1000]
explainer = shap.TreeExplainer(model, ...)
explanation = explainer(...)
margin = model.predict(sample, output_margin=True)
gap = np.abs(explanation.base_values + ... - margin).max()
print(
  f"Valeur de base : {explanation.base_values[0]:.3f} log-odds ; "
  f"écart d'additivité maximal : {gap:.1e}"
)
shap.plots.beeswarm(...)
shap.plots.bar(...)
```

### Solution

In [ ]:
import shap

sample = X_test.iloc[:1000]
explainer = shap.TreeExplainer(model, X_train.sample(100, random_state=0))
explanation = explainer(sample)
margin = model.predict(sample, output_margin=True)
gap = np.abs(explanation.base_values + explanation.values.sum(axis=1) - margin).max()
print(
  f"Valeur de base : {explanation.base_values[0]:.3f} log-odds ; "
  f"écart d'additivité maximal : {gap:.1e}"
)
shap.plots.beeswarm(explanation)
shap.plots.bar(explanation)

## 5. Un client mal prédit face à un client bien prédit

Parmi les clients de test qui ont fait défaut, trouvez `worst`, celui que le modèle jugeait le moins risqué, et `best`, celui qu'il jugeait le plus risqué : leurs positions dans `X_test`. Affichez leurs variables côte à côte, puis la cascade (*waterfall*) SHAP de chacun. Pourquoi le modèle s'est-il trompé sur `worst` ? L'erreur vient-elle du modèle (un défaut qu'il aurait pu voir dans ces variables) ou des données (rien, dans ses variables, ne le distingue d'un bon payeur) ?

**Documentation utile**

- [`numpy.flatnonzero`](https://numpy.org/doc/stable/reference/generated/numpy.flatnonzero.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.flatnonzero(y_test == 1)` : les positions des clients qui ont fait défaut ; `proba[defaulters].argmin()` donne, parmi eux, celle du moins risqué selon le modèle
- `X_test.iloc[[worst, best]].T` : leurs variables, une colonne par client 
- `explainer(X_test.iloc[[worst, best]])` : les valeurs SHAP des deux clients ; `shap.plots.waterfall(cases[0])`, puis `cases[1]`
- La cascade part de la valeur de base, en bas, et ajoute la contribution de chaque variable jusqu'à la sortie du modèle pour ce client, en haut

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
defaulters = np.flatnonzero(y_test == 1)
worst = defaulters[proba[defaulters].argmin()]
best = ...
print(
  f"Probabilité de défaut : {proba[worst]:.3f} pour worst, {proba[best]:.3f} pour best"
)
display(X_test.iloc[[worst, best]].T)
cases = explainer(...)
shap.plots.waterfall(cases[0])
shap.plots.waterfall(...)
```

### Solution

In [ ]:
defaulters = np.flatnonzero(y_test == 1)
worst = defaulters[proba[defaulters].argmin()]
best = defaulters[proba[defaulters].argmax()]
print(
  f"Probabilité de défaut : {proba[worst]:.3f} pour worst, {proba[best]:.3f} pour best"
)
display(X_test.iloc[[worst, best]].T)
cases = explainer(X_test.iloc[[worst, best]])
shap.plots.waterfall(cases[0])
shap.plots.waterfall(cases[1])

## 6. LIME face à SHAP sur le même client

Expliquez la prédiction de `worst` avec LIME : un `LimeTabularExplainer` construit sur `X_train`, puis `explain_instance` avec les 5 variables les plus importantes. Comparez avec les 5 plus grandes valeurs SHAP de ce client, en valeur absolue. Relancez LIME avec un autre `random_state` : l'explication change-t-elle ? Pourquoi LIME et SHAP ne donnent-ils pas les mêmes nombres ?

**Documentation utile**

- [`LimeTabularExplainer`](https://lime-ml.readthedocs.io/en/latest/lime.html#lime.lime_tabular.LimeTabularExplainer)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `LimeTabularExplainer(X_train.to_numpy(), feature_names=list(X.columns), class_names=["sain", "défaut"], mode="classification", random_state=0)` : LIME travaille sur des tableaux NumPy, et découpe chaque variable en quartiles
- `lime_explainer.explain_instance(X_test.to_numpy()[worst], model.predict_proba, num_features=5)` ; `.as_list()` renvoie les conditions et leurs poids, sur la probabilité de défaut
- `pd.Series(cases[0].values, index=X.columns)` : les valeurs SHAP de `worst` (calculées à l'exercice 5) ; `shap_values[shap_values.abs().nlargest(5).index]` : les 5 plus grandes
- LIME ajuste un modèle linéaire local, sur des variables découpées en intervalles, à la probabilité ; SHAP répartit exactement la sortie en log-odds entre les variables

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from lime.lime_tabular import LimeTabularExplainer

for seed in (0, 1):
  lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(),
    feature_names=list(X.columns),
    class_names=["sain", "défaut"],
    mode="classification",
    random_state=seed,
  )
  lime_explanation = lime_explainer.explain_instance(
    ..., model.predict_proba, num_features=...
  )
  print(f"LIME, random_state={seed} :")
  for condition, weight in lime_explanation.as_list():
    print(f"  {condition:30} {weight:+.3f}")

shap_values = pd.Series(cases[0].values, index=X.columns)
print("SHAP :")
print(shap_values[...].round(3))
```

### Solution

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

for seed in (0, 1):
  lime_explainer = LimeTabularExplainer(
    X_train.to_numpy(),
    feature_names=list(X.columns),
    class_names=["sain", "défaut"],
    mode="classification",
    random_state=seed,
  )
  lime_explanation = lime_explainer.explain_instance(
    X_test.to_numpy()[worst], model.predict_proba, num_features=5
  )
  print(f"LIME, random_state={seed} :")
  for condition, weight in lime_explanation.as_list():
    print(f"  {condition:30} {weight:+.3f}")

shap_values = pd.Series(cases[0].values, index=X.columns)
print("SHAP :")
print(shap_values[shap_values.abs().nlargest(5).index].round(3))

## 7. Importance par permutation et dépendance partielle

Calculez l'importance par permutation de chaque variable sur le jeu de test, comme la baisse de l'AUC ROC, et comparez son classement à celui du graphique en barres de SHAP. Tracez ensuite la dépendance partielle et les courbes ICE de `delay_sep` et de `limit` : l'effet de `limit` est-il le même pour tous les clients ?

**Documentation utile**

- [`PartialDependenceDisplay.from_estimator`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.PartialDependenceDisplay.html#sklearn.inspection.PartialDependenceDisplay.from_estimator)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=0)` (une quinzaine de secondes) ; `pd.Series(result.importances_mean, index=X.columns).nlargest(8)` : les 8 plus importantes
- `PartialDependenceDisplay.from_estimator(model, X_test, features=["delay_sep", "limit"], kind="both", subsample=300, random_state=0)` : `kind="both"` trace les courbes ICE, une par client, et leur moyenne, la dépendance partielle ; `subsample=300` ne garde que 300 clients, pour que les courbes restent lisibles

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

result = permutation_importance(
  model, X_test, y_test, scoring=..., n_repeats=10, random_state=0
)
print(pd.Series(result.importances_mean, index=X.columns).nlargest(8).round(4))

PartialDependenceDisplay.from_estimator(
  model,
  X_test,
  features=[..., ...],
  kind=...,
  subsample=300,
  random_state=0,
)
plt.show()
```

### Solution

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

result = permutation_importance(
  model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=0
)
print(pd.Series(result.importances_mean, index=X.columns).nlargest(8).round(4))

PartialDependenceDisplay.from_estimator(
  model,
  X_test,
  features=["delay_sep", "limit"],
  kind="both",
  subsample=300,
  random_state=0,
)
plt.show()

## 8. Tranches des données : où le modèle coûte le plus

Les décisions sont désormais celles du seuil choisi en validation croisée (`decisions` : 1 pour un refus). Écrivez `slice_report(slices)`, qui renvoie un tableau d'une ligne par tranche des clients de test : sa proportion, son taux de défaut, le rappel (la part des défauts refusés), le taux de refus, le coût moyen par client et l'impact potentiel. Le crédit n'a pas de performance humaine de référence : l'écart se mesure ici à un modèle parfait, de coût nul, et vaut donc le coût par client ; l'impact potentiel est, comme dans les slides, l'écart multiplié par la proportion de la tranche. Appliquez `slice_report` au retard de septembre (`delay`), à l'âge (`age`) et aux études (`education`). Quelle tranche travailler en premier ? Qu'a-t-elle de commun avec le client `worst` ?

**Documentation utile**

- [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [`recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html)

In [ ]:
decisions = pd.Series(tuned.predict(X_test), index=X_test.index)
delay = pd.cut(
  X_test.delay_sep, [-3, 0, 1, 9], labels=["à jour", "1 mois", "2 mois et plus"]
)
age = pd.cut(
  X_test.age, [20, 30, 40, 50, 80], labels=["21-30", "31-40", "41-50", "51+"]
)
education = X_test.education.map(
  {1: "2e-3e cycle", 2: "université", 3: "lycée"}
).fillna("autre ou inconnu")


def slice_report(slices: pd.Series) -> pd.DataFrame: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices}).groupby("slice", observed=True)` : un groupe par tranche ; `observed=True` ignore les catégories vides
- Pour chaque `(name, group)` du `groupby` : `len(group) / len(data)`, `group.y.mean()`, `recall_score(group.y, group.refused)`, `group.refused.mean()`, `cost(group.y, group.refused) / len(group)`
- Un dictionnaire de lignes, `{name: {"proportion": ..., ...}}`, devient un tableau avec `pd.DataFrame.from_dict(rows, orient="index")`
- Pour vérifier : l'impact potentiel de la tranche « à jour » est d'environ 0,46

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import recall_score


def slice_report(slices: pd.Series) -> pd.DataFrame:
  data = pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices})
  rows = {}
  for name, group in data.groupby("slice", observed=True):
    proportion = ...
    per_client = cost(group.y, group.refused) / len(group)
    rows[name] = {
      "proportion": proportion,
      "taux de défaut": group.y.mean(),
      "rappel": ...,
      "taux de refus": ...,
      "coût par client": per_client,
      "impact": ...,
    }
  return pd.DataFrame.from_dict(rows, orient="index").round(3)


for slices in (delay, age, education):
  display(slice_report(slices))
```

### Solution

In [ ]:
from sklearn.metrics import recall_score


def slice_report(slices: pd.Series) -> pd.DataFrame:
  data = pd.DataFrame({"y": y_test, "refused": decisions, "slice": slices})
  rows = {}
  for name, group in data.groupby("slice", observed=True):
    proportion = len(group) / len(data)
    per_client = cost(group.y, group.refused) / len(group)
    rows[name] = {
      "proportion": proportion,
      "taux de défaut": group.y.mean(),
      "rappel": recall_score(group.y, group.refused),
      "taux de refus": group.refused.mean(),
      "coût par client": per_client,
      "impact": per_client * proportion,
    }
  return pd.DataFrame.from_dict(rows, orient="index").round(3)


for slices in (delay, age, education):
  display(slice_report(slices))

## 9. Équité : les erreurs par groupe

Avec un `MetricFrame` de Fairlearn, comparez entre hommes et femmes le rappel, le taux de refus et le taux de faux positifs des décisions (la part des clients sains refusés), puis faites de même par tranche d'âge (`age`). Quels écarts voyez-vous, et lequel pèse sur des clients qui auraient remboursé ? `sex` est une variable du modèle : la retirer suffirait-il à supprimer ces écarts ?

**Documentation utile**

- [`false_positive_rate`](https://fairlearn.org/main/api_reference/generated/fairlearn.metrics.false_positive_rate.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_test.sex.map({1: "homme", 2: "femme"})` : des noms de groupes lisibles
- `MetricFrame(metrics={"rappel": recall_score, ...}, y_true=y_test, y_pred=decisions, sensitive_features=sex)` : `metrics` associe un nom à chaque fonction `(y_true, y_pred)` ; `selection_rate` et `false_positive_rate` viennent de `fairlearn.metrics`
- `frame.by_group` : une ligne par groupe ; `frame.difference()` : le plus grand écart entre groupes, pour chaque métrique

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from fairlearn.metrics import MetricFrame, false_positive_rate, selection_rate

sex = X_test.sex.map({1: "homme", 2: "femme"})
metrics = {
  "rappel": recall_score,
  "taux de refus": ...,
  "faux positifs": ...,
}
for groups in (sex, age):
  frame = MetricFrame(
    metrics=metrics, y_true=y_test, y_pred=..., sensitive_features=...
  )
  display(frame.by_group.round(3))
  print(frame.difference().round(3))
```

### Solution

In [ ]:
from fairlearn.metrics import MetricFrame, false_positive_rate, selection_rate

sex = X_test.sex.map({1: "homme", 2: "femme"})
metrics = {
  "rappel": recall_score,
  "taux de refus": selection_rate,
  "faux positifs": false_positive_rate,
}
for groups in (sex, age):
  frame = MetricFrame(
    metrics=metrics, y_true=y_test, y_pred=decisions, sensitive_features=groups
  )
  display(frame.by_group.round(3))
  print(frame.difference().round(3))

## Pour aller plus loin

- Un modèle interprétable par construction, souvent aussi bon que XGBoost sur des données tabulaires : l'[EBM d'InterpretML](https://interpret.ml/docs/ebm.html) (`ExplainableBoostingClassifier`, `pip install interpret`)
- Des contrefactuels, les plus petits changements qui auraient fait accorder le crédit à un client refusé, en ne touchant qu'aux variables sur lesquelles il peut agir : [DiCE](https://github.com/interpretml/DiCE) (`pip install dice-ml`)